# UNet-Strong Tumor-Only 224x224

Run the stronger UNet recipe for BTXRD raw or preprocessed tumor-only data using the new repo pipeline. Set `DATA_MODE` to `raw` for `E0s`, or `preprocessed` for `E1as`.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/lehngoc/BTXRD-LViT.git"
BRANCH = "model/e1-unet-baseline"
REPO_ROOT = Path("/kaggle/working/BTXRD-LViT")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
import torch
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Select Run

- `DATA_MODE = "raw"`: `E0s`, raw images + raw converted masks, same seed42 split.
- `DATA_MODE = "preprocessed"`: `E1as`, preprocessed images/masks, same seed42 split.

In [ ]:
# Change only these paths if Kaggle input names are different.
DATA_MODE = "raw"  # "raw" or "preprocessed"

RAW_DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-raw/btxrd-raw")
PREPROCESSED_DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-preprocessed-dataset/btxrd-preprocessed")

IMAGE_SIZE = 224
BATCH_SIZE = 4
ACCUMULATION_STEPS = 2
NUM_WORKERS = 2
EPOCHS = 200
LR = 3e-4
PATIENCE = 100

if DATA_MODE not in {"raw", "preprocessed"}:
    raise ValueError(f"Unsupported DATA_MODE: {DATA_MODE}")

if DATA_MODE == "raw":
    DATA_ROOT = RAW_DATA_ROOT
    RUN_NAME = "E0s_unet_strong_raw_224_tumor_only"
else:
    DATA_ROOT = PREPROCESSED_DATA_ROOT
    RUN_NAME = "E1as_unet_strong_preprocessed_224_tumor_only"

OUTPUT_DIR = Path(f"/kaggle/working/experiments/{RUN_NAME}")
MANIFEST_DIR = Path(f"/kaggle/working/{RUN_NAME}_manifests")
RUNTIME_CONFIG = Path(f"/kaggle/working/{RUN_NAME}.yaml")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_MODE =", DATA_MODE)
print("DATA_ROOT =", DATA_ROOT)
print("RUN_NAME =", RUN_NAME)
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
if DATA_MODE == "raw":
    required = [
        DATA_ROOT / "data/raw/images",
        DATA_ROOT / "data/processed/masks",
        DATA_ROOT / "data/exports/btxrd_preprocessed/train.csv",
        DATA_ROOT / "data/exports/btxrd_preprocessed/val.csv",
        DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv",
    ]
else:
    required = [
        DATA_ROOT / "data/processed/images_preprocessed",
        DATA_ROOT / "data/processed/masks_preprocessed",
        DATA_ROOT / "data/exports/btxrd_preprocessed/train.csv",
        DATA_ROOT / "data/exports/btxrd_preprocessed/val.csv",
        DATA_ROOT / "data/exports/btxrd_preprocessed/test.csv",
    ]

for path in required:
    print(path, "->", path.exists())

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

## Build Manifests

For raw mode, the seed42 split CSVs are reused but paths are rewritten to raw images and raw converted masks. For preprocessed mode, the exported CSVs are used directly.

In [ ]:
import pandas as pd

def find_relative_raw_image(image_id: str) -> str:
    stem = Path(str(image_id)).stem
    candidates = [
        Path("data/raw/images") / str(image_id),
        Path("data/raw/images") / f"{stem}.jpeg",
        Path("data/raw/images") / f"{stem}.jpg",
        Path("data/raw/images") / f"{stem}.png",
    ]
    for rel_path in candidates:
        if (DATA_ROOT / rel_path).exists():
            return str(rel_path)
    raise FileNotFoundError(f"Missing raw image for {image_id}")

def find_relative_raw_mask(image_id: str) -> str:
    stem = Path(str(image_id)).stem
    candidates = [
        Path("data/processed/masks") / f"{stem}.png",
        Path("data/processed/masks") / f"{stem}.jpg",
        Path("data/processed/masks") / f"{stem}.jpeg",
    ]
    for rel_path in candidates:
        if (DATA_ROOT / rel_path).exists():
            return str(rel_path)
    raise FileNotFoundError(f"Missing raw mask for {image_id}")

def write_manifest(split: str) -> Path:
    src_csv = DATA_ROOT / f"data/exports/btxrd_preprocessed/{split}.csv"
    if DATA_MODE == "preprocessed":
        return src_csv

    dst_csv = MANIFEST_DIR / f"{split}.csv"
    df = pd.read_csv(src_csv)
    df["image_path"] = df["image_id"].map(find_relative_raw_image)
    df["mask_path"] = df["image_id"].map(find_relative_raw_mask)
    df.to_csv(dst_csv, index=False)
    return dst_csv

train_csv = write_manifest("train")
val_csv = write_manifest("val")
test_csv = write_manifest("test")

for split, path in [("train", train_csv), ("val", val_csv), ("test", test_csv)]:
    df = pd.read_csv(path)
    tumor_count = int(df["tumor"].astype(int).sum())
    print(f"{split}: rows={len(df)} tumor={tumor_count} normal={len(df)-tumor_count} -> {path}")

In [ ]:
import yaml

cfg = {
    "experiment": {"name": RUN_NAME},
    "data": {
        "root_dir": str(DATA_ROOT),
        "tumor_only": True,
        "train_csv": str(train_csv),
        "val_csv": str(val_csv),
        "test_csv": str(test_csv),
    },
    "model": {
        "name": "unet",
        "in_channels": 3,
        "out_channels": 1,
        "base_channels": 64,
    },
    "training": {
        "seed": 42,
        "device": "cuda",
        "image_size": IMAGE_SIZE,
        "image_mean": [0.0, 0.0, 0.0],
        "image_std": [1.0, 1.0, 1.0],
        "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS,
        "num_workers": NUM_WORKERS,
        "epochs": EPOCHS,
        "learning_rate": LR,
        "weight_decay": 0.0,
        "optimizer": "adam",
        "loss_type": "legacy_weighted_dice_bce",
        "bce_weight": 0.5,
        "dice_weight": 0.5,
        "foreground_weight": 0.3,
        "background_weight": 0.7,
        "early_stopping_patience": PATIENCE,
        "label_fraction": 1.0,
        "text_column": "text_lvit_prompt",
        "output_dir": str(OUTPUT_DIR),
        "augmentation": {"enabled": True, "type": "legacy_lvit"},
        "scheduler": {"name": "cosine_warm_restarts", "t_0": 10, "t_mult": 1, "eta_min": 1e-4},
    },
    "metrics": {"threshold": 0.5, "min_fp_area_ratio": 0.001},
}

RUNTIME_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(RUNTIME_CONFIG.read_text())

In [ ]:
!python src/training/smoke_test_model_pipeline.py --config {RUNTIME_CONFIG} --samples-per-split 8

In [ ]:
!python src/training/train_unet.py --config {RUNTIME_CONFIG} --device cuda

In [ ]:
best_ckpt = OUTPUT_DIR / "best.pt"
assert best_ckpt.exists(), best_ckpt

!python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split val --device cuda --output {OUTPUT_DIR / 'val_metrics.json'}
!python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split test --device cuda --output {OUTPUT_DIR / 'test_metrics.json'}

In [ ]:
import json

for name in ["best_summary.json", "val_metrics.json", "test_metrics.json"]:
    path = OUTPUT_DIR / name
    print("\n===", name, "===")
    print(json.dumps(json.loads(path.read_text()), indent=2))

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
sweep = []
for thr in thresholds:
    out = OUTPUT_DIR / f"val_metrics_thr{int(thr * 100):02d}.json"
    !python src/training/evaluate_unet.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split val --device cuda --threshold {thr} --output {out}
    metrics = json.loads(out.read_text())
    sweep.append({
        "threshold": thr,
        "tumor_dice": metrics["tumor_dice"],
        "tumor_iou": metrics["tumor_iou"],
        "normal_pred_area_ratio": metrics["normal_pred_area_ratio"],
        "normal_fp_image_rate": metrics["normal_fp_image_rate"],
    })

(OUTPUT_DIR / "threshold_sweep_metrics.json").write_text(json.dumps(sweep, indent=2), encoding="utf-8")
print(json.dumps(sweep, indent=2))

In [ ]:
!python src/training/visualize_unet_predictions.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split val --device cuda --output-dir {OUTPUT_DIR / 'visual_checks_val'} --max-tumor 24 --max-normal 0
!python src/training/visualize_unet_predictions.py --config {RUNTIME_CONFIG} --checkpoint {best_ckpt} --split test --device cuda --output-dir {OUTPUT_DIR / 'visual_checks_test'} --max-tumor 24 --max-normal 0

In [ ]:
import zipfile

metrics_zip = Path(f"/kaggle/working/{RUN_NAME}_metrics_only.zip")
wanted = [
    "history.csv",
    "best_summary.json",
    "val_metrics.json",
    "test_metrics.json",
    "threshold_sweep_metrics.json",
    "config.json",
]
with zipfile.ZipFile(metrics_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in wanted:
        path = OUTPUT_DIR / name
        if path.exists():
            z.write(path, arcname=name)
    for path in sorted(OUTPUT_DIR.glob("val_metrics_thr*.json")):
        z.write(path, arcname=path.name)

full_zip = Path(f"/kaggle/working/{RUN_NAME}_full_artifacts.zip")
with zipfile.ZipFile(full_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname=str(path.relative_to(OUTPUT_DIR)))

print("metrics zip:", metrics_zip, metrics_zip.stat().st_size / 1024 / 1024, "MB")
print("full zip:", full_zip, full_zip.stat().st_size / 1024 / 1024, "MB")